# RAC Vision: detector de cabezas con y sin casco

Este notebook entrena un detector YOLOv8n que encuentra **cabezas** en la imagen y dice si tienen
casco o no, desde cualquier ángulo (de frente, de costado, de espaldas o con la cara tapada).
Reemplaza el esquema anterior de "detectar la cara → recortar → clasificar".

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.
Después ejecutá las celdas en orden. No cierres la pestaña mientras entrena.

Al final se descargan dos archivos:
- `detector_casco.onnx`: va en la raíz del proyecto (al lado de `modelo.keras`). La app lo usa con OpenCV, sin PyTorch.
- `detector_casco.pt`: guardalo para reentrenar en el futuro.

Datos: [Hard Hat Detection](https://huggingface.co/datasets/dgural/hard-hat-detection) (5000 fotos, licencia CC0)
más tus fotos de `con_casco/` y `sin_casco/`.

## 1. Instalar librerías y verificar la GPU

In [ ]:
!nvidia-smi -L
!pip install -q "ultralytics>=8.3,<8.4" huggingface_hub

## 2. Descargar el dataset público (≈1.3 GB)

In [ ]:
from huggingface_hub import snapshot_download
SRC = snapshot_download("dgural/hard-hat-detection", repo_type="dataset", local_dir="/content/hardhat")
print("Descargado en", SRC)

## 3. Convertir a formato YOLO

Clases del detector: `0 = casco` (cabeza con casco) y `1 = sin_casco` (cabeza sin casco).
Las cajas de "persona" del dataset original se descartan.

In [ ]:
import json, random, shutil
from collections import Counter
from pathlib import Path

SRC = Path("/content/hardhat")
DST = Path("/content/casco_yolo")
CLASES = ["casco", "sin_casco"]
MAPA = {"helmet": 0, "head": 1}

samples = json.load(open(SRC / "samples.json"))["samples"]
random.seed(0)
random.shuffle(samples)
n_val = int(len(samples) * 0.15)

etiquetas_vistas, conteo = Counter(), Counter()
for i, s in enumerate(samples):
    split = "val" if i < n_val else "train"
    img = SRC / s["filepath"]
    if not img.exists():
        img = SRC / "data" / Path(s["filepath"]).name
    (DST / "images" / split).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / split).mkdir(parents=True, exist_ok=True)
    shutil.copy(img, DST / "images" / split / img.name)
    lineas = []
    for d in (s.get("ground_truth") or {}).get("detections", []):
        etiquetas_vistas[d["label"]] += 1
        c = MAPA.get(d["label"].lower())
        if c is None:
            continue
        x, y, w, h = d["bounding_box"]  # FiftyOne: esquina superior izquierda, relativo
        lineas.append(f"{c} {x + w / 2:.6f} {y + h / 2:.6f} {w:.6f} {h:.6f}")
        conteo[CLASES[c]] += 1
    (DST / "labels" / split / (img.stem + ".txt")).write_text("\n".join(lineas))

(DST / "data.yaml").write_text(
    f"path: {DST}\ntrain: images/train\nval: images/val\nnames:\n  0: casco\n  1: sin_casco\n")
print(len(samples), "imágenes |", n_val, "de validación")
print("Etiquetas originales:", dict(etiquetas_vistas))
print("Cajas usadas:", dict(conteo))

## 4. Subir tus fotos

Subí `fotos_rac.zip` (adentro tiene las carpetas `con_casco/` y `sin_casco/`).
El 70% se usa para ajustar el modelo a tu cámara y el 30% queda aparte para medir si funciona.

In [ ]:
import zipfile
from google.colab import files

subido = files.upload()
zipfile.ZipFile(next(iter(subido))).extractall("/content/rac")

fotos = []
for real in ("con_casco", "sin_casco"):
    for p in sorted(Path("/content/rac", real).glob("*.jpg")):
        fotos.append((p, real))
random.seed(1)
random.shuffle(fotos)
corte = int(len(fotos) * 0.7)
fotos_ajuste, fotos_prueba = fotos[:corte], fotos[corte:]
print(len(fotos_ajuste), "para ajustar |", len(fotos_prueba), "para probar")

## 5. Entrenar el modelo base con el dataset público

In [ ]:
from ultralytics import YOLO

base = YOLO("yolov8n.pt")
base.train(data=str(DST / "data.yaml"), epochs=50, imgsz=640, batch=32, patience=15,
           project="/content/runs", name="base", exist_ok=True)

## 6. Medir con tus fotos de prueba

Cada foto cuenta como **sin casco** si aparece alguna cabeza sin casco, **con casco** si solo hay
cabezas con casco, y **nada** si no encontró ninguna cabeza.

In [ ]:
def evaluar(ruta_modelo, conf=0.35):
    modelo = YOLO(ruta_modelo)
    tabla, ok = Counter(), 0
    for p, real in fotos_prueba:
        r = modelo.predict(str(p), conf=conf, imgsz=640, verbose=False)[0]
        clases = {int(c) for c in r.boxes.cls}
        pred = "sin_casco" if 1 in clases else "con_casco" if 0 in clases else "nada"
        tabla[(real, pred)] += 1
        ok += pred == real
    print(f"Aciertos: {ok}/{len(fotos_prueba)}")
    for (real, pred), n in sorted(tabla.items()):
        print(f"  real={real:10s} predicho={pred:10s} {n}")

evaluar("/content/runs/base/weights/best.pt")

## 7. Ajustar el modelo a tu cámara

El modelo base marca la cabeza en tus fotos de ajuste y la clase sale de la carpeta
(`con_casco/` o `sin_casco/`). Esas cajas se suman al dataset y se reentrena unas épocas más.
Las fotos donde no encontró ninguna cabeza se listan y se dejan afuera.

In [ ]:
modelo_base = YOLO("/content/runs/base/weights/best.pt")
sin_caja = []
for p, real in fotos_ajuste:
    r = modelo_base.predict(str(p), conf=0.2, imgsz=640, verbose=False)[0]
    if len(r.boxes) == 0:
        sin_caja.append(p.name)
        continue
    mejor = int(r.boxes.conf.argmax())
    cx, cy, w, h = r.boxes.xywhn[mejor].tolist()
    c = 0 if real == "con_casco" else 1
    for k in range(3):  # se repite para que pesen más frente a las 5000 fotos públicas
        nombre = f"rac_{real}_{p.stem}_{k}"
        shutil.copy(p, DST / "images" / "train" / f"{nombre}.jpg")
        (DST / "labels" / "train" / f"{nombre}.txt").write_text(f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
print("Sin cabeza detectada (no se usan):", sin_caja)

ajustado = YOLO("/content/runs/base/weights/best.pt")
ajustado.train(data=str(DST / "data.yaml"), epochs=20, imgsz=640, batch=32, lr0=0.002,
               project="/content/runs", name="ajustado", exist_ok=True)
evaluar("/content/runs/ajustado/weights/best.pt")

## 8. Exportar y descargar

Si en el paso 7 los aciertos bajaron respecto del paso 6, cambiá `ELEGIDO` por `"base"`.

In [ ]:
ELEGIDO = "ajustado"
final = YOLO(f"/content/runs/{ELEGIDO}/weights/best.pt")
onnx = final.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(onnx, "/content/detector_casco.onnx")
shutil.copy(f"/content/runs/{ELEGIDO}/weights/best.pt", "/content/detector_casco.pt")
files.download("/content/detector_casco.onnx")
files.download("/content/detector_casco.pt")